<a href="https://colab.research.google.com/github/Nityakothavari7/EmberMind/blob/main/05_memory_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

PROJECT = "EMBER-X"

INPUTS = [
    "therapy_index.faiss",
    "therapy_metadata.pkl",
    "ember_qwen_lora",
]

OUTPUTS = [
    "user_memory.json"
]

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

print(os.path.exists("/content/drive/MyDrive"))

Mounted at /content/drive
True


In [ ]:
import json
import os

MEMORY_FILE = "/content/drive/MyDrive/user_memory.json"

if not os.path.exists(MEMORY_FILE):

    with open(MEMORY_FILE, "w") as f:
        json.dump([], f)

print("Memory Store Ready")
print("Exists:", os.path.exists(MEMORY_FILE))

Memory Store Ready
Exists: True


In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding Model Loaded")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding Model Loaded


In [ ]:
HIGH_DISTRESS = [

    "suicide",
    "kill myself",
    "self harm",

    "hopeless",
    "meaningless",

    "depressed",
    "depression",

    "panic attack",

    "abuse",
    "trauma"
]

MEDIUM_DISTRESS = [

    "anxiety",
    "worried",

    "lonely",
    "loneliness",

    "breakup",
    "divorce",

    "grief",
    "loss",

    "job loss",
    "unemployed"
]


def calculate_importance(text):

    text = text.lower()

    score = 0.1

    for word in HIGH_DISTRESS:

        if word in text:
            score += 0.4

    for word in MEDIUM_DISTRESS:

        if word in text:
            score += 0.2

    return min(score, 1.0)

In [ ]:
def detect_memory_type(text):

    text = text.lower()

    if any(x in text for x in [
        "suicide",
        "kill myself",
        "self harm"
    ]):
        return "crisis"

    if any(x in text for x in [
        "hopeless",
        "meaningless",
        "depressed"
    ]):
        return "high_distress"

    if any(x in text for x in [
        "anxiety",
        "panic",
        "worried"
    ]):
        return "anxiety"

    if any(x in text for x in [
        "breakup",
        "relationship",
        "divorce"
    ]):
        return "relationship"

    return "general"

In [ ]:
def create_summary(text):

    if len(text) < 120:
        return text

    return text[:120] + "..."

In [ ]:
from datetime import datetime

def save_memory(user_message):

    importance = calculate_importance(
        user_message
    )

    if importance < 0.3:

        print("Memory not important enough")
        return

    embedding = embedding_model.encode(
        user_message
    ).tolist()

    memory = {

        "timestamp": str(datetime.now()),

        "user_message": user_message,

        "memory_summary": create_summary(
            user_message
        ),

        "importance_score": importance,

        "memory_type": detect_memory_type(
            user_message
        ),

        "embedding": embedding
    }

    with open(MEMORY_FILE, "r") as f:
        memories = json.load(f)

    memories.append(memory)

    with open(MEMORY_FILE, "w") as f:
        json.dump(
            memories,
            f,
            indent=4
        )

    print("Memory Saved")

In [ ]:
save_memory(
    "I feel hopeless and life feels meaningless."
)

save_memory(
    "My girlfriend broke up with me and I feel lonely."
)

save_memory(
    "Good morning."
)

Memory Saved
Memory Saved
Memory not important enough


In [ ]:
import faiss
import numpy as np
import json

with open(MEMORY_FILE, "r") as f:
    memories = json.load(f)

print("Memories:", len(memories))

memory_vectors = []

for memory in memories:

    memory_vectors.append(
        memory["embedding"]
    )

memory_vectors = np.array(
    memory_vectors,
    dtype=np.float32
)

dimension = memory_vectors.shape[1]

memory_index = faiss.IndexFlatIP(
    dimension
)

faiss.normalize_L2(
    memory_vectors
)

memory_index.add(
    memory_vectors
)

print("Indexed Memories:", memory_index.ntotal)

Memories: 2
Indexed Memories: 2


In [ ]:
faiss.write_index(
    memory_index,
    "/content/drive/MyDrive/memory_index.faiss"
)

print("Memory Index Saved")

Memory Index Saved


In [ ]:
!pip install faiss-cpu


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 61.2 MB/s eta 0:00:00


In [ ]:
def retrieve_memories(
    query,
    top_k=3
):

    with open(MEMORY_FILE, "r") as f:
        memories = json.load(f)

    actual_k = min(
        top_k,
        len(memories)
    )

    query_embedding = embedding_model.encode(
        [query]
    )

    query_embedding = np.array(
        query_embedding,
        dtype=np.float32
    )

    faiss.normalize_L2(
        query_embedding
    )

    scores, indices = memory_index.search(
        query_embedding,
        actual_k
    )

    results = []

    for score, idx in zip(
        scores[0],
        indices[0]
    ):

        results.append({

            "similarity": float(score),

            "memory": memories[idx]
        })

    return results

In [ ]:
results = retrieve_memories(
    "I still feel hopeless."
)

for item in results:

    print("=" * 80)

    print(
        "Similarity:",
        round(item["similarity"], 3)
    )

    print(
        "Type:",
        item["memory"]["memory_type"]
    )

    print(
        "Importance:",
        item["memory"]["importance_score"]
    )

    print(
        "Summary:",
        item["memory"]["memory_summary"]
    )

Similarity: 0.736
Type: high_distress
Importance: 0.9
Summary: I feel hopeless and life feels meaningless.
Similarity: 0.469
Type: general
Importance: 0.30000000000000004
Summary: My girlfriend broke up with me and I feel lonely.


Context Builder

Input:
[memory_index.faiss,
therapy_index.faiss,
therapy_metadata.pkl,
user_memory.json]

Output:
RAG Context

In [ ]:
import pickle

with open(
    "/content/drive/MyDrive/therapy_metadata.pkl",
    "rb"
) as f:

    therapy_docs = pickle.load(f)

print("Therapy Docs:", len(therapy_docs))

therapy_index = faiss.read_index(
    "/content/drive/MyDrive/therapy_index.faiss"
)

Therapy Docs: 16994


In [ ]:
def retrieve_therapy_examples(
    query,
    top_k=3
):

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    )

    scores, indices = therapy_index.search(
        query_embedding.astype(np.float32),
        top_k
    )

    results = []

    for score, idx in zip(
        scores[0],
        indices[0]
    ):

        results.append({

            "score": float(score),

            "document": therapy_docs[idx]
        })

    return results

In [ ]:
results = retrieve_therapy_examples(
    "I feel hopeless and lonely."
)

for r in results:

    print("="*80)

    print("Score:", r["score"])

    print(r["document"][:1200])

    print()

Score: 0.824393630027771

User:
I've tried joining online support groups, but I find it hard to connect with others there as well. I've also tried reaching out to old friends, but it feels like they're too busy with their own lives to really listen or understand. I've been feeling hopeless and helpless, and I don't know what to do.

Therapist:
I'm sorry to hear that you've been feeling hopeless and helpless, and that your efforts to connect with others have been challenging. It's essential to remember that these feelings are common, especially during times of uncertainty and isolation. However, it's also crucial to recognize that you have the power to take steps towards improving your mental and emotional wellbeing. This could include engaging in activities that bring you joy, such as reading, painting, or listening to music. It could also involve reaching out to mental health professionals for guidance and support. Remember, it's okay to ask for help, and there are resources available

In [ ]:
def build_context(
    user_message,
    memory_k=3,
    therapy_k=3
):

    memories = retrieve_memories(
        user_message,
        memory_k
    )

    therapies = retrieve_therapy_examples(
        user_message,
        therapy_k
    )

    context = ""

    context += "\n[USER MEMORY]\n\n"

    for item in memories:

        context += (
            f"- {item['memory']['memory_summary']}\n"
        )

    context += "\n[THERAPY EXAMPLES]\n\n"

    for idx, item in enumerate(
        therapies,
        start=1
    ):

        example = item["document"][:1000]

        context += (
            f"Example {idx}:\n"
        )

        context += example

        context += "\n\n"

    context += (
        "\n[CURRENT USER]\n\n"
    )

    context += user_message

    return context

In [ ]:
context = build_context(
    "I still feel hopeless and lonely."
)

print(context[:5000])


[USER MEMORY]

- I feel hopeless and life feels meaningless.
- My girlfriend broke up with me and I feel lonely.

[THERAPY EXAMPLES]

Example 1:

User:
I've been feeling empty and disconnected from the world around me for months now. It's like I'm just going through the motions, but nothing seems to bring me joy or excitement anymore. I can't help but think about the past and all the things I've missed out on. My friends are all moving on with their lives, getting married, having children, and I feel like I'm stuck in place. I can't shake this feeling of sadness and hopelessness.

Therapist:
I understand that you've been feeling disconnected and empty for quite some time now, and it's understandable that you're feeling sad and hopeless about the present moment. It's important to remember that feelings are temporary and that they don't define your worth or your potential for happiness. I'd like to help you explore the root causes of these feelings and work together to develop strategie